# Offline embedding — InsuranceRAG

Runs the bi-encoder on a Colab GPU because a 0.6B model over ~2,900 chunks is hours on CPU.
Nothing else about the pipeline moves: chunks in, vectors out, Postgres stays on your laptop.

**In** `data/chunks/*.jsonl` (from `scripts/ingest.py`), zipped and uploaded.
**Out** `data/embeddings/{doc_id}.npz`, holding `ids` and `vectors`, downloaded back.

Then locally: `python scripts/index.py --embeddings data/embeddings`.

In [1]:
!pip install -q sentence-transformers
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 1. Upload the chunks

Zip `data/chunks` locally first:

```powershell
Compress-Archive -Path data\chunks\*.jsonl -DestinationPath chunks.zip -Force
```

In [3]:
from google.colab import files

files.upload()  # pick chunks.zip
!rm -rf chunks embeddings && mkdir -p chunks embeddings
!unzip -q -j chunks.zip -d chunks
!ls -la chunks

KeyboardInterrupt: 

## 2. Load the encoder

fp16 on a T4 halves memory and roughly doubles throughput; the vectors are cast back to
float32 before saving so pgvector receives the same precision it would locally.

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"  # must equal settings.bi_encoder_model
BATCH_SIZE = 32

model = SentenceTransformer(
    MODEL_ID, device="cuda", model_kwargs={"torch_dtype": "float16"}
)
model.max_seq_length = 1024  # chunks are capped at 800 reference tokens
print(model.get_sentence_embedding_dimension())

## 3. Embed, one document at a time

Per-document files, written as each finishes — a Colab disconnect then costs one document,
not the whole run. Re-running skips whatever is already on disk.

In [ ]:
import json
from pathlib import Path

import numpy as np

CHUNKS = Path("chunks")
OUT = Path("embeddings")

for path in sorted(CHUNKS.glob("*.jsonl")):
    target = OUT / f"{path.stem}.npz"
    if target.exists():
        print(f"{path.stem:<40} cached")
        continue

    records = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    vectors = model.encode(
        [r["text"] for r in records],
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype("float32")

    np.savez(target, ids=np.array([r["chunk_id"] for r in records]), vectors=vectors)
    print(f"{path.stem:<40} {vectors.shape}")

## 4. Download

Unzip into `data/embeddings/` locally, then run `python scripts/index.py --embeddings data/embeddings`.

In [ ]:
!zip -q -r embeddings.zip embeddings
files.download("embeddings.zip")